In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[16, 32, 64, 128]):
        super(AttentionUNet, self).__init__()
        self.encoder1 = self.conv_block(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder2 = self.conv_block(features[0], features[1])
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder3 = self.conv_block(features[1], features[2])
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder4 = self.conv_block(features[2], features[3])
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = self.conv_block(features[3], features[3]*2)

        self.up4 = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=features[3], F_l=features[3], F_int=features[3]//2)
        self.decoder4 = self.decoder_conv_block(features[3]*2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=features[2], F_l=features[2], F_int=features[2]//2)
        self.decoder3 = self.decoder_conv_block(features[2]*2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=features[1], F_l=features[1], F_int=features[1]//2)
        self.decoder2 = self.decoder_conv_block(features[1]*2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=features[0], F_l=features[0], F_int=features[0]//2)
        self.decoder1 = self.decoder_conv_block(features[0]*2, features[0])

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def decoder_conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        p1 = self.pool1(e1)

        e2 = self.encoder2(p1)
        p2 = self.pool2(e2)

        e3 = self.encoder3(p2)
        p3 = self.pool3(e3)

        e4 = self.encoder4(p3)
        p4 = self.pool4(e4)

        b = self.bottleneck(p4)

        up4 = self.up4(b)
        att4 = self.att4(g=up4, x=e4)
        d4 = torch.cat([up4, att4], dim=1)
        d4 = self.decoder4(d4)

        up3 = self.up3(d4)
        att3 = self.att3(g=up3, x=e3)
        d3 = torch.cat([up3, att3], dim=1)
        d3 = self.decoder3(d3)

        up2 = self.up2(d3)
        att2 = self.att2(g=up2, x=e2)
        d2 = torch.cat([up2, att2], dim=1)
        d2 = self.decoder2(d2)

        up1 = self.up1(d2)
        att1 = self.att1(g=up1, x=e1)
        d1 = torch.cat([up1, att1], dim=1)
        d1 = self.decoder1(d1)

        out = self.final_conv(d1)
        return out
    
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

def iou_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = (pred + target - pred * target).sum(dim=(1, 2, 3)) + epsilon
    iou = intersection / union
    return iou.mean()

def precision_score(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum(dim=(1, 2, 3))
    fp = (pred * (1 - target)).sum(dim=(1, 2, 3)) + epsilon
    precision = tp / (tp + fp)
    return precision.mean()

def recall_score(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum(dim=(1, 2, 3))
    fn = ((1 - pred) * target).sum(dim=(1, 2, 3)) + epsilon
    recall = tp / (tp + fn)
    return recall.mean()

class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0.5).float()

        return image, mask

In [2]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. لود مدل آموزشدیده
model = AttentionUNet().to(device)
model.load_state_dict(torch.load('attention_unet_fold_2.pth', map_location=device))
model.eval()

# 2. آمادهسازی دادهها
# آدرس فولدرها
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))

# تقسیم K-Fold (همانند زمان آموزش)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(images_list))
fold = 2  # انتخاب فولد 2 (مطمئن شوید با ذخیرهسازی مدل مطابقت دارد)
train_idx, val_idx = splits[fold]  # برای تست روی دادههای آموزشی فولد 2

# ایجاد دیتاست و دیتالودر برای دادههای آموزشی فولد 2
test_dataset = CorneaDataset(
    images_list=[images_list[i] for i in train_idx],
    masks_list=[masks_list[i] for i in train_idx],
    transform=transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 3. ارزیابی مدل و محاسبه معیارها
total_dice = 0.0
total_iou = 0.0
total_precision = 0.0
total_recall = 0.0
samples_to_visualize = []

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        
        # محاسبه معیارها
        dice = dice_coefficient(outputs, masks)
        iou = iou_coefficient(outputs, masks)
        precision = precision_score(outputs, masks)
        recall = recall_score(outputs, masks)
        
        total_dice += dice.item()
        total_iou += iou.item()
        total_precision += precision.item()
        total_recall += recall.item()
        
        # ذخیره نمونهها برای نمایش
        if idx < 3:  # نمایش 3 نمونه اول
            samples_to_visualize.append((
                images.cpu().squeeze().permute(1, 2, 0).numpy(),
                torch.sigmoid(outputs).cpu().squeeze().numpy(),
                masks.cpu().squeeze().numpy()
            ))

# محاسبه میانگین معیارها
num_samples = len(test_loader)
print(f"\nنتایج ارزیابی روی {num_samples} نمونه:")
print(f"Dice Score: {total_dice / num_samples:.4f}")
print(f"IoU: {total_iou / num_samples:.4f}")
print(f"Precision: {total_precision / num_samples:.4f}")
print(f"Recall: {total_recall / num_samples:.4f}")

# 4. نمایش نمونهها
plt.figure(figsize=(15, 10))
for i, (img, pred, mask) in enumerate(samples_to_visualize):
    # پیشپردازش پیشبینی
    pred = (pred > 0.5).astype(np.float32)
    
    # نمایش تصاویر
    plt.subplot(3, 3, i*3 + 1)
    plt.imshow(img)
    plt.title('ورودی (تصویر اصلی)')
    plt.axis('off')
    
    plt.subplot(3, 3, i*3 + 2)
    plt.imshow(pred, cmap='gray')
    plt.title('خروجی مدل (پیشبینی)')
    plt.axis('off')
    
    plt.subplot(3, 3, i*3 + 3)
    plt.imshow(mask, cmap='gray')
    plt.title('ماسک واقعی')
    plt.axis('off')

plt.tight_layout()
plt.show()


In [19]:
import os
import glob
import numpy as np
import torch
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

def postprocess_corneal_mask(predicted_mask, morph_kernel_size=13, blur_kernel_size=13):
    mask = predicted_mask.astype(np.uint8) * 255
    
    # 1. اعمال عملیات مورفولوژیک با کرنل بزرگتر
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size))
    morph_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    morph_mask = cv2.morphologyEx(morph_mask, cv2.MORPH_OPEN, kernel)
    
    # 2. افزودن Gaussian Blur
    blurred = cv2.GaussianBlur(morph_mask, (blur_kernel_size, blur_kernel_size), 0)
    
    # 3. استخراج کانتور از تصویر بلور شده
    contours, _ = cv2.findContours(blurred, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return mask
    
    largest_contour = max(contours, key=cv2.contourArea)
    
    if len(largest_contour) < 5:
        return mask
    
    # 4. فیت کردن بیضی
    ellipse = cv2.fitEllipse(largest_contour)
    h, w = mask.shape[:2]
    processed_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(processed_mask, ellipse, 255, thickness=-1)
    
    return processed_mask

# کلاس دیتاست (باید با تعریف اصلی شما مطابقت داشته باشد)
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        image = Image.open(self.images_list[idx]).convert('RGB')
        mask = Image.open(self.masks_list[idx]).convert('L')
        
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
            
        return image, mask

# توابع متریک (باید با تعریف اصلی شما مطابقت داشته باشد)
def dice_coefficient(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def iou_coefficient(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def precision_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    true_positives = (pred * target).sum()
    predicted_positives = pred.sum()
    return true_positives / (predicted_positives + 1e-5)

def recall_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    true_positives = (pred * target).sum()
    actual_positives = target.sum()
    return true_positives / (actual_positives + 1e-5)

# بخش اصلی کد
if __name__ == "__main__":
    # تنظیمات دستگاه
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. لود مدل آموزشدیده
    model = AttentionUNet().to(device)  # فرض می‌کنیم کلاس AttentionUNet تعریف شده است
    model.load_state_dict(torch.load('attention_unet_fold_2.pth', map_location=device))
    model.eval()

    # 2. آماده‌سازی داده‌ها
    images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
    labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

    images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
    masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))

    # تقسیم K-Fold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(images_list))
    fold = 2
    train_idx, val_idx = splits[fold]

    # ایجاد دیتاست و دیتالودر
    test_dataset = CorneaDataset(
        images_list=[images_list[i] for i in train_idx],
        masks_list=[masks_list[i] for i in train_idx],
        transform=transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])
    )
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # 3. ارزیابی و نمایش نتایج
    total_dice = 0.0
    total_iou = 0.0
    total_precision = 0.0
    total_recall = 0.0
    samples_to_visualize = []

    with torch.no_grad():
        for idx, (images, masks) in enumerate(test_loader):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            
            # محاسبه معیارها
            dice = dice_coefficient(outputs, masks)
            iou = iou_coefficient(outputs, masks)
            precision = precision_score(outputs, masks)
            recall = recall_score(outputs, masks)
            
            total_dice += dice.item()
            total_iou += iou.item()
            total_precision += precision.item()
            total_recall += recall.item()
            
            # ذخیره نمونه‌ها برای نمایش
            if idx < 3:
                samples_to_visualize.append((
                    images.cpu().squeeze().permute(1, 2, 0).numpy(),
                    torch.sigmoid(outputs).cpu().squeeze().numpy(),
                    masks.cpu().squeeze().numpy()
                ))

    # نمایش معیارها
    num_samples = len(test_loader)
    print(f"\nنتایج ارزیابی روی {num_samples} نمونه:")
    print(f"Dice Score: {total_dice / num_samples:.4f}")
    print(f"IoU: {total_iou / num_samples:.4f}")
    print(f"Precision: {total_precision / num_samples:.4f}")
    print(f"Recall: {total_recall / num_samples:.4f}")

    plt.figure(figsize=(15, 10))
    for i, (img, pred, mask) in enumerate(samples_to_visualize):
        # پردازش پیش‌بینی با پارامترهای جدید
        raw_pred = (pred > 0.5).astype(np.uint8)
        
        # فراخوانی تابع با تنظیمات بهبودیافته
        processed_pred = postprocess_corneal_mask(
            raw_pred,
            morph_kernel_size=9,  # افزایش اندازه کرنل مورفولوژیک
            blur_kernel_size=5    # افزودن Gaussian Blur
        )
        
        # نرمال‌سازی تصویر
        img = (img - img.min()) / (img.max() - img.min())
        
        # نمایش نتایج
        plt.subplot(3, 3, i*3 + 1)
        plt.imshow(img)
        plt.title('ورودی (تصویر اصلی)')
        plt.axis('off')
        
        plt.subplot(3, 3, i*3 + 2)
        plt.imshow(processed_pred, cmap='gray')
        plt.title('پیش‌بینی پس‌پردازش شده')
        plt.axis('off')
        
        plt.subplot(3, 3, i*3 + 3)
        plt.imshow(mask, cmap='gray')
        plt.title('ماسک واقعی')
        plt.axis('off')
    

In [46]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
import cv2
# تابع Hough Circle Transform (از پاسخ قبلی)
def process_mask_with_hough_circle(mask, dp=1, min_dist=40, param1=100, param2=10, min_radius=10, max_radius=100):
    global idx  # برای استفاده از idx در چاپ
    if mask.dtype != np.uint8:
        mask = mask.astype(np.uint8)
    
    # پیش‌پردازش: بلور گاوسی ملایم‌تر برای حفظ لبه‌ها
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    # تقویت حلقه با Dilate برای ضخیم‌تر کردن
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    
    # پر کردن حفره‌ها با مورفولوژی
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=4)  # پر کردن مرکز
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)   # حذف نویز
    
    # پر کردن حفره‌ها با Flood Fill یا کانتور
    mask_filled = mask.copy()
    h, w = mask.shape[:2]
    mask_temp = np.zeros((h+2, w+2), np.uint8)
    cv2.floodFill(mask_filled, mask_temp, (0, 0), 255)  # پر کردن پس‌زمینه
    mask_filled = cv2.bitwise_not(mask_filled)  # معکوس کردن برای گرفتن مناطق پرشده
    mask = cv2.bitwise_or(mask, mask_filled)  # ترکیب با ماسک اصلی
    
    # ذخیره ماسک پیش‌پردازش‌شده
    cv2.imwrite(f"test_preprocessed_mask_{idx}.png", mask)
    
    processed_mask = np.zeros_like(mask, dtype=np.uint8)
    
    circles = cv2.HoughCircles(
        mask,
        cv2.HOUGH_GRADIENT,
        dp=dp,
        minDist=min_dist,
        param1=param1,
        param2=param2,
        minRadius=min_radius,
        maxRadius=max_radius
    )
    
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        print(f"نمونه {idx}: تعداد دایره‌های تشخیص داده‌شده: {len(circles)}")
        print(f"مختصات و شعاع دایره‌ها: {circles}")
        for (x, y, r) in circles:
            cv2.circle(processed_mask, (x, y), r, 255, -1)
    else:
        print(f"نمونه {idx}: هیچ دایره‌ای تشخیص داده نشد.")
    
    return processed_mask

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. لود مدل آموزش‌دیده
model = AttentionUNet().to(device)  # فرض بر این است که کلاس AttentionUNet تعریف شده است
model.load_state_dict(torch.load('attention_unet_fold_2.pth', map_location=device))
model.eval()

# 2. آماده‌سازی داده‌ها
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))

# تقسیم K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf.split(images_list))
fold = 2
train_idx, val_idx = splits[fold]

# ایجاد دیتاست و دیتالودر
test_dataset = CorneaDataset(  # فرض بر این است که کلاس CorneaDataset تعریف شده است
    images_list=[images_list[i] for i in train_idx],
    masks_list=[masks_list[i] for i in train_idx],
    transform=transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 3. ارزیابی مدل و محاسبه معیارها
total_dice = 0.0
total_iou = 0.0
total_precision = 0.0
total_recall = 0.0
total_dice_processed = 0.0  # برای ماسک‌های پردازش‌شده
samples_to_visualize = []

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        
        # پیش‌پردازش خروجی مدل
        pred = torch.sigmoid(outputs).cpu().squeeze().numpy()  # تبدیل به احتمال
        pred_binary = (pred > 0.5).astype(np.uint8) * 255  # تبدیل به ماسک باینری
        
        # اعمال Hough Circle Transform
        pred_processed = process_mask_with_hough_circle(
            pred_binary,
            dp=1,
            min_dist=20,
            param1=50,
            param2=30,
            min_radius=10,
            max_radius=100
        )
        
        # تبدیل ماسک پردازش‌شده به فرمت مناسب برای محاسبه معیارها
        pred_processed_tensor = torch.from_numpy(pred_processed / 255.0).float().unsqueeze(0).unsqueeze(0).to(device)
        
        # محاسبه معیارها برای خروجی اصلی
        dice = dice_coefficient(outputs, masks)
        iou = iou_coefficient(outputs, masks)
        precision = precision_score(outputs, masks)
        recall = recall_score(outputs, masks)
        
        # محاسبه معیار Dice برای ماسک پردازش‌شده
        dice_processed = dice_coefficient(pred_processed_tensor, masks)
        
        total_dice += dice.item()
        total_iou += iou.item()
        total_precision += precision.item()
        total_recall += recall.item()
        total_dice_processed += dice_processed.item()
        
        # ذخیره نمونه‌ها برای نمایش
        if idx < 3:  # نمایش 3 نمونه اول
            samples_to_visualize.append((
                images.cpu().squeeze().permute(1, 2, 0).numpy(),
                pred_binary / 255.0,  # خروجی اصلی مدل
                pred_processed / 255.0,  # خروجی پردازش‌شده
                masks.cpu().squeeze().numpy()
            ))

# محاسبه میانگین معیارها
num_samples = len(test_loader)
print(f"\nنتایج ارزیابی روی {num_samples} نمونه:")
print("برای خروجی اصلی مدل:")
print(f"Dice Score: {total_dice / num_samples:.4f}")
print(f"IoU: {total_iou / num_samples:.4f}")
print(f"Precision: {total_precision / num_samples:.4f}")
print(f"Recall: {total_recall / num_samples:.4f}")
print("\nبرای ماسک‌های پردازش‌شده با Hough Circle Transform:")
print(f"Dice Score: {total_dice_processed / num_samples:.4f}")

# 4. نمایش نمونه‌ها
plt.figure(figsize=(20, 10))
for i, (img, pred, pred_processed, mask) in enumerate(samples_to_visualize):
    # نمایش تصاویر
    plt.subplot(3, 4, i*4 + 1)
    plt.imshow(img)
    plt.title('ورودی (تصویر اصلی)')
    plt.axis('off')
    
    plt.subplot(3, 4, i*4 + 2)
    plt.imshow(pred, cmap='gray')
    plt.title('خروجی مدل (پیش‌بینی)')
    plt.axis('off')
    
    plt.subplot(3, 4, i*4 + 3)
    plt.imshow(pred_processed, cmap='gray')
    plt.title('ماسک پردازش‌شده (Hough)')
    plt.axis('off')
    
    plt.subplot(3, 4, i*4 + 4)
    plt.imshow(mask, cmap='gray')
    plt.title('ماسک واقعی')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [49]:
import cv2
from skimage.metrics import mean_squared_error

# افزودن توابع جدید برای پردازش دایره
def apply_hough_circle(mask_np):
    # تبدیل به تصویر 8 بیتی
    mask_8bit = (mask_np * 255).astype(np.uint8)
    
    # پارامترهای Hough Circle (میتوانید تنظیم کنید)
    circles = cv2.HoughCircles(mask_8bit, 
                                cv2.HOUGH_GRADIENT, 
                                dp=1, 
                                minDist=50,    # کاهش فاصله حداقلی بین مراکز
                                param1=50,     # آستانه گرادیان (میتواند ثابت بماند)
                                param2=15,     # کاهش آستانه تشخیص دایره
                                minRadius=20,  # تنظیم بر اساس اندازه واقعی قرنیه
                                maxRadius=60
                            )
    
    # ایجاد ماسک خالی
    h, w = mask_np.shape
    circle_mask = np.zeros((h, w), dtype=np.float32)
    
    if circles is not None:
        circles = np.uint16(np.around(circles))
        # انتخاب اولین دایره (فرض میکنیم فقط یک قرنیه وجود دارد)
        circle = circles[0][0]
        cv2.circle(circle_mask, 
                  (circle[0], circle[1]), 
                  circle[2], 
                  1, 
                  thickness=-1)
    
    return circle_mask

# قسمت ارزیابی را اصلاح میکنیم
total_dice_original = 0.0
total_dice_processed = 0.0
total_iou_original = 0.0
total_iou_processed = 0.0
samples_to_visualize = []

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        
        # تبدیل خروجی مدل به ماسک باینری
        pred_np = (torch.sigmoid(outputs).cpu().squeeze().numpy())
        pred_mask = (pred_np > 0.5).astype(np.float32)
        
        # اعمال Hough Circle Transform
        processed_mask = apply_hough_circle(pred_mask)
        
        # تبدیل به تنسور برای محاسبه معیارها
        processed_tensor = torch.from_numpy(processed_mask).unsqueeze(0).to(device)
        
        # محاسبه معیارها برای خروجی اصلی
        dice_original = dice_coefficient(outputs, masks)
        iou_original = iou_coefficient(outputs, masks)
        
        # محاسبه معیارها برای خروجی پردازش شده
        dice_processed = dice_coefficient(processed_tensor, masks)
        iou_processed = iou_coefficient(processed_tensor, masks)
        
        # جمع معیارها
        total_dice_original += dice_original.item()
        total_dice_processed += dice_processed.item()
        total_iou_original += iou_original.item()
        total_iou_processed += iou_processed.item()
        
        # ذخیره نمونهها برای نمایش
        if idx < 3:
            samples_to_visualize.append((
                images.cpu().squeeze().permute(1, 2, 0).numpy(),
                pred_mask,
                processed_mask,
                masks.cpu().squeeze().numpy()
            ))

# محاسبه میانگین معیارها
num_samples = len(test_loader)
print("\nنتایج ارزیابی:")
print(f"[Original] Dice: {total_dice_original/num_samples:.4f}, IoU: {total_iou_original/num_samples:.4f}")
print(f"[Processed] Dice: {total_dice_processed/num_samples:.4f}, IoU: {total_iou_processed/num_samples:.4f}")

# نمایش نمونهها با خروجی پردازش شده
plt.figure(figsize=(15, 12))
for i, (img, pred, processed, mask) in enumerate(samples_to_visualize):
    plt.subplot(4, 3, i*4 + 1)
    plt.imshow(img)
    plt.title('تصویر ورودی')
    plt.axis('off')
    
    plt.subplot(4, 3, i*4 + 2)
    plt.imshow(pred, cmap='gray')
    plt.title('خروجی مدل')
    plt.axis('off')
    
    plt.subplot(4, 3, i*4 + 3)
    plt.imshow(processed, cmap='gray')
    plt.title('خروجی پردازش شده')
    plt.axis('off')
    
    plt.subplot(4, 3, i*4 + 4)
    plt.imshow(mask, cmap='gray')
    plt.title('ماسک واقعی')
    plt.axis('off')

plt.tight_layout()
plt.show()